In [2]:
%pip install -qU "langchain[google-genai]"

Note: you may need to restart the kernel to use updated packages.


In [3]:
import getpass
import os

GOOGLE_API_KEY = os.environ.get("GEMINI_API_KEY")

if not GOOGLE_API_KEY:
  GOOGLE_API_KEY = getpass.getpass("Enter API key for Google Gemini: ")

from langchain.chat_models import init_chat_model

llm = init_chat_model("gemini-2.5-flash", model_provider="google_genai", api_key=GOOGLE_API_KEY)

## Embeddings

In [4]:
%pip install langchain-huggingface sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
import torch
import os
from dotenv import load_dotenv

# Load environment variables for HuggingFace token
load_dotenv()

huggingface_api_key = os.environ.get("HUGGINGFACE_API_TOKEN")
if huggingface_api_key:
    os.environ["HUGGINGFACE_HUB_TOKEN"] = huggingface_api_key

# Use the same multilingual embedding model from main.py
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🔧 Using device: {device}")

# Initialize the multilingual embedding model (same as main.py)
embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-large-instruct",
    model_kwargs={'device': device}
    # model_kwargs={'device': device, 'token': huggingface_api_key}
)

print("✅ HuggingFace embeddings initialized successfully!")
print(f"Model: intfloat/multilingual-e5-large-instruct")
print("🔄 Model will be downloaded locally (first time only)")
print(f"🇮🇳 Testing with Indian languages!")

# Test the embeddings with multiple Indian languages
indian_test_texts = [
    "यह एक परीक्षण वाक्य है।",  # Hindi
    "এটি একটি পরীক্ষার বাক্য।",  # Bengali  
    "இது ஒரு சோதனை வாக்கியம்.",  # Tamil
    "ಇದು ಒಂದು ಪರೀಕ್ಷಾ ವಾಕ್ಯವಾಗಿದೆ.",  # Kannada
    "ଏହା ଏକ ପରୀକ୍ଷା ବାକ୍ୟ।",  # Odia
    "हे एक चाचणी वाक्य आहे.",  # Marathi
    "આ એક પરીક્ષણ વાક્ય છે.",  # Gujarati
    "ਇਹ ਇੱਕ ਟੈਸਟ ਵਾਕ ਹੈ।",  # Punjabi
    "ఇది ఒక పరీక్ష వాక్యం.",  # Telugu
    "ഇത് ഒരു പരീക്ഷണ വാക്യമാണ്."  # Malayalam
]

print("\n🧪 Testing Indian language embeddings:")
for i, text in enumerate(indian_test_texts):
    # Add instruction prefix like in main.py
    prefixed_text = f"Represent this document for semantic search: {text}"
    test_embedding = embeddings.embed_query(prefixed_text)
    
    # Detect language name
    languages = ["Hindi", "Bengali", "Tamil", "Kannada", "Odia", "Marathi", "Gujarati", "Punjabi", "Telugu", "Malayalam"]
    
    print(f"📊 {languages[i]}: {text}")
    print(f"   Embedding dimension: {len(test_embedding)}")
    print(f"   First 5 values: {test_embedding[:5]}")
    print()

print("🎯 All Indian languages processed successfully!")

🔧 Using device: cuda
✅ HuggingFace embeddings initialized successfully!
Model: intfloat/multilingual-e5-large-instruct
🔄 Model will be downloaded locally (first time only)
🇮🇳 Testing with Indian languages!

🧪 Testing Indian language embeddings:
📊 Hindi: यह एक परीक्षण वाक्य है।
   Embedding dimension: 1024
   First 5 values: [0.021396588534116745, 0.010903053916990757, 0.008092647418379784, -0.05982087180018425, 0.04329632595181465]

📊 Bengali: এটি একটি পরীক্ষার বাক্য।
   Embedding dimension: 1024
   First 5 values: [0.02998528815805912, -0.0017081241821870208, -0.0036847570445388556, -0.05284345895051956, 0.034686144441366196]

📊 Tamil: இது ஒரு சோதனை வாக்கியம்.
   Embedding dimension: 1024
   First 5 values: [0.02440408617258072, 0.027767416089773178, 0.001557129668071866, -0.05000105872750282, 0.04855753853917122]

📊 Kannada: ಇದು ಒಂದು ಪರೀಕ್ಷಾ ವಾಕ್ಯವಾಗಿದೆ.
   Embedding dimension: 1024
   First 5 values: [0.015329491347074509, 0.01667376421391964, -0.0015496768755838275, -0.044709026813

In [6]:
# embeddings = HuggingFaceEmbeddings(model_name="NovaSearch/stella_en_1.5B_v5")

# print("✅ HuggingFace embeddings initialized successfully!")
# print(f"Model: sentence-transformers/all-mpnet-base-v2")
# print("🔄 Model will be downloaded locally (first time only)")

# # Test the embeddings
# test_text = "This is a test sentence for embeddings."
# test_embedding = embeddings.embed_query(test_text)
# print(f"📊 Embedding dimension: {len(test_embedding)}")
# print(f"🧪 Test embedding (first 5 values): {test_embedding[:5]}")

In [7]:
# Test Hindi Q&A simulation
hindi_test_doc = "भारत दुनिया का सबसे बड़ा लोकतंत्र है। यहाँ 28 राज्य और 8 केंद्र शासित प्रदेश हैं। भारत में 22 आधिकारिक भाषाएँ हैं।"

hindi_question = "भारत में कितने राज्य हैं?"

print("🇮🇳 Hindi RAG Simulation:")
print(f"📄 Document: {hindi_test_doc}")
print(f"❓ Question: {hindi_question}")

# Get embeddings for both
doc_embedding = embeddings.embed_query(f"Represent this document for semantic search: {hindi_test_doc}")
question_embedding = embeddings.embed_query(hindi_question)

# Calculate similarity (cosine similarity)
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity([doc_embedding], [question_embedding])[0][0]
print(f"🔍 Semantic similarity: {similarity:.4f}")

# Simulate what Gemini would receive
print(f"\n🤖 What Gemini receives:")
print(f"Context: {hindi_test_doc}")
print(f"Question: {hindi_question}")
print(f"Expected Response: भारत में 28 राज्य हैं।")

🇮🇳 Hindi RAG Simulation:
📄 Document: भारत दुनिया का सबसे बड़ा लोकतंत्र है। यहाँ 28 राज्य और 8 केंद्र शासित प्रदेश हैं। भारत में 22 आधिकारिक भाषाएँ हैं।
❓ Question: भारत में कितने राज्य हैं?
🔍 Semantic similarity: 0.8944

🤖 What Gemini receives:
Context: भारत दुनिया का सबसे बड़ा लोकतंत्र है। यहाँ 28 राज्य और 8 केंद्र शासित प्रदेश हैं। भारत में 22 आधिकारिक भाषाएँ हैं।
Question: भारत में कितने राज्य हैं?
Expected Response: भारत में 28 राज्य हैं।


In [8]:
# Test Hindi Q&A simulation
hindi_test_doc = "भारत दुनिया का सबसे बड़ा लोकतंत्र है। यहाँ 28 राज्य और 8 केंद्र शासित प्रदेश हैं। भारत में 22 आधिकारिक भाषाएँ हैं।"

hindi_question = "भारत में कितने राज्य हैं?"

print("🇮🇳 Hindi RAG Simulation:")
print(f"📄 Document: {hindi_test_doc}")
print(f"❓ Question: {hindi_question}")

# Get embeddings for both
doc_embedding = embeddings.embed_query(f"Represent this document for semantic search: {hindi_test_doc}")
question_embedding = embeddings.embed_query(hindi_question)

# Calculate similarity (cosine similarity)
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity([doc_embedding], [question_embedding])[0][0]
print(f"🔍 Semantic similarity: {similarity:.4f}")

# Simulate what Gemini would receive
print(f"\n🤖 What Gemini receives:")
print(f"Context: {hindi_test_doc}")
print(f"Question: {hindi_question}")

# ==================== NEW: CONNECT TO GEMINI ====================

# Create the same prompt structure as main.py
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. You answer questions based only on the provided context. If the answer is not in the context, say 'I don't know'."),
    ("user", "Question: {question}\n\nContext: {context}")
])

# Format the prompt with our Hindi data
formatted_prompt = prompt.format(
    question=hindi_question,
    context=hindi_test_doc
)

print(f"\n📝 Formatted prompt sent to Gemini:")
print(formatted_prompt)

# Get actual response from Gemini
print(f"\n🚀 Getting response from Gemini...")
try:
    response = llm.invoke(formatted_prompt)
    print(f"✅ Gemini Response: {response.content}")
except Exception as e:
    print(f"❌ Error: {e}")

# ==================== BONUS: TEST CROSS-LANGUAGE ====================

print(f"\n" + "="*50)
print("🌍 CROSS-LANGUAGE TEST")
print("="*50)

# Ask in English about Hindi content
english_question = "How many states are there in India?"

cross_lang_prompt = prompt.format(
    question=english_question,
    context=hindi_test_doc
)

print(f"📄 Hindi Context: {hindi_test_doc}")
print(f"❓ English Question: {english_question}")

try:
    cross_response = llm.invoke(cross_lang_prompt)
    print(f"✅ Gemini Cross-Language Response: {cross_response.content}")
except Exception as e:
    print(f"❌ Error: {e}")

🇮🇳 Hindi RAG Simulation:
📄 Document: भारत दुनिया का सबसे बड़ा लोकतंत्र है। यहाँ 28 राज्य और 8 केंद्र शासित प्रदेश हैं। भारत में 22 आधिकारिक भाषाएँ हैं।
❓ Question: भारत में कितने राज्य हैं?
🔍 Semantic similarity: 0.8944

🤖 What Gemini receives:
Context: भारत दुनिया का सबसे बड़ा लोकतंत्र है। यहाँ 28 राज्य और 8 केंद्र शासित प्रदेश हैं। भारत में 22 आधिकारिक भाषाएँ हैं।
Question: भारत में कितने राज्य हैं?

📝 Formatted prompt sent to Gemini:
System: You are a helpful assistant. You answer questions based only on the provided context. If the answer is not in the context, say 'I don't know'.
Human: Question: भारत में कितने राज्य हैं?

Context: भारत दुनिया का सबसे बड़ा लोकतंत्र है। यहाँ 28 राज्य और 8 केंद्र शासित प्रदेश हैं। भारत में 22 आधिकारिक भाषाएँ हैं।

🚀 Getting response from Gemini...
✅ Gemini Response: भारत में 28 राज्य हैं।

🌍 CROSS-LANGUAGE TEST
📄 Hindi Context: भारत दुनिया का सबसे बड़ा लोकतंत्र है। यहाँ 28 राज्य और 8 केंद्र शासित प्रदेश हैं। भारत में 22 आधिकारिक भाषाएँ हैं।
❓ Englis

In [9]:
hindi_text = """
विस्तृत लेख: भारत  विविधता और समृद्धि की भूमि

भारत दुनिया का सबसे बड़ा लोकतंत्र है और अपनी जनसंख्या, संस्कृति और ऐतिहासिक योगदान के कारण एक विशेष स्थान रखता है। वर्तमान में भारत की जनसंख्या लगभग 140 करोड़ से अधिक है, जो इसे विश्व का सबसे अधिक जनसंख्या वाला देश बनाती है। प्रशासनिक दृष्टि से यहाँ 28 राज्य और 8 केंद्र शासित प्रदेश हैं। इसकी राजधानी नई दिल्ली है जबकि मुंबई को देश का सबसे बड़ा शहर और वित्तीय राजधानी माना जाता है।

भाषा और विविधता

भारत की पहचान उसकी भाषाई विविधता से होती है। संविधान द्वारा 22 भाषाओं को आधिकारिक दर्जा दिया गया है, जिनमें हिंदी और अंग्रेजी मुख्य हैं। इसके अलावा, सैकड़ों बोलियाँ और क्षेत्रीय भाषाएँ बोली जाती हैं। यह भाषाई विविधता भारतीय समाज की गहराई और लोक-संस्कृति को दर्शाती है।

ऐतिहासिक परिप्रेक्ष्य

भारत का इतिहास हजारों वर्षों पुराना है। सिंधु घाटी सभ्यता (3300 ई.पू.) दुनिया की सबसे प्राचीन शहरी सभ्यताओं में से एक थी। इसके बाद वैदिक काल, मौर्य साम्राज्य (चंद्रगुप्त और अशोक), गुप्त काल (स्वर्ण युग) और मुगल साम्राज्य ने कला, विज्ञान और वास्तुकला में नई ऊँचाइयाँ प्रदान कीं।
17वीं शताब्दी में ब्रिटिश ईस्ट इंडिया कंपनी के आगमन ने औपनिवेशिक शासन का मार्ग प्रशस्त किया। लगभग दो शताब्दियों तक चले संघर्ष के बाद 1947 में स्वतंत्रता प्राप्त हुई। स्वतंत्रता संग्राम के महानायक – महात्मा गांधी, जवाहरलाल नेहरू, सरदार वल्लभभाई पटेल, सुभाष चंद्र बोस और भगत सिंह जैसे क्रांतिकारी – देश की आज़ादी के स्तंभ बने।

भूगोल और प्राकृतिक विविधता

भारत का भूगोल अत्यंत विविध है। उत्तर में हिमालय पर्वत श्रृंखला विश्व की सबसे ऊँची चोटियों का घर है। पश्चिम में थार रेगिस्तान, पूर्वोत्तर में घने वन और पर्वत, तथा दक्षिण में लंबा तटीय क्षेत्र इसकी विशिष्टता बढ़ाते हैं। भारत की प्रमुख नदियाँ – गंगा, यमुना, ब्रह्मपुत्र, गोदावरी, नर्मदा और कावेरी – कृषि और जीवन के लिए महत्वपूर्ण हैं।

भारत की जलवायु भी विविध है – उत्तर में शीतल हिमालयी जलवायु से लेकर दक्षिण में उष्णकटिबंधीय जलवायु तक। मानसून कृषि के लिए जीवनरेखा है।

संस्कृति और धर्म

भारत को धर्मों की जन्मभूमि कहा जाता है। हिंदू धर्म, बौद्ध धर्म, जैन धर्म और सिख धर्म यहीं से उत्पन्न हुए। यहाँ ईसाई, इस्लाम, यहूदी और पारसी धर्म के अनुयायी भी शांति से रहते हैं।
त्योहार भारतीय जीवन का महत्वपूर्ण अंग हैं। दीवाली, होली, ईद, क्रिसमस, पोंगल, बैसाखी और गणेश चतुर्थी पूरे देश में उल्लास से मनाए जाते हैं।

भारतीय कला, संगीत और नृत्य ने भी विश्व को प्रभावित किया है। भरतनाट्यम, कथक, ओडिसी, कथकली जैसे शास्त्रीय नृत्य तथा हिंदुस्तानी और कर्नाटक संगीत भारतीय संस्कृति के गौरवशाली स्तंभ हैं।

अर्थव्यवस्था और विकास

भारत की अर्थव्यवस्था तेजी से विकसित हो रही है। वर्तमान में यह दुनिया की पाँचवीं सबसे बड़ी अर्थव्यवस्था है। यह एक मिश्रित अर्थव्यवस्था है, जहाँ सरकारी और निजी क्षेत्र दोनों का योगदान है।

कृषि: रोजगार का सबसे बड़ा स्रोत।

उद्योग: विनिर्माण, ऑटोमोबाइल, स्टील, कपड़ा।

सेवा क्षेत्र: IT, बैंकिंग, टेलीकॉम, पर्यटन।

विशेष रूप से सूचना प्रौद्योगिकी और सॉफ्टवेयर उद्योग ने भारत को विश्व स्तर पर अग्रणी बना दिया है। बेंगलुरु को "भारत की सिलिकॉन वैली" कहा जाता है।

शिक्षा और विज्ञान

भारत शिक्षा के क्षेत्र में भी प्रगति कर रहा है। IITs, IIMs, AIIMS और अन्य विश्वविद्यालय विश्व स्तर पर मान्यता प्राप्त हैं।
इसरो (ISRO) ने चंद्रयान, मंगलयान और हाल ही में चंद्रयान-3 जैसे अभियानों के माध्यम से अंतरिक्ष विज्ञान में भारत को अग्रणी बनाया है।

खेल और मनोरंजन

भारत में क्रिकेट सबसे लोकप्रिय खेल है। भारतीय क्रिकेट टीम ने विश्व कप सहित कई बड़ी उपलब्धियाँ हासिल की हैं।
इसके अलावा, हॉकी (राष्ट्रीय खेल), बैडमिंटन, कबड्डी और कुश्ती भी लोकप्रिय हैं।
सिनेमा – विशेषकर बॉलीवुड – भारत की सांस्कृतिक पहचान का अहम हिस्सा है।

सामाजिक संरचना

भारत की सबसे बड़ी शक्ति इसकी एकता में विविधता है। विभिन्न धर्मों, भाषाओं और संस्कृतियों के लोग यहाँ साथ रहते हैं। भारतीय संविधान प्रत्येक नागरिक को समानता, स्वतंत्रता और न्याय का अधिकार प्रदान करता है।

निष्कर्ष

भारत परंपरा और आधुनिकता का संगम है। जहाँ एक ओर यहाँ की जड़ें प्राचीन संस्कृति से जुड़ी हैं, वहीं दूसरी ओर यह आधुनिक विज्ञान, तकनीक और अर्थव्यवस्था के क्षेत्र में वैश्विक नेतृत्व कर रहा है।
"""

In [10]:
# Simple Hindi RAG using LangChain built-in functions (PostgreSQL like main.py)

# 1. Install dependencies (already in requirements.txt)
# langchain-postgres, langchain-google-genai, langchain-huggingface are already installed

# 2. Initialize models (same as main.py)
import getpass
import os
from dotenv import load_dotenv

load_dotenv()
GOOGLE_API_KEY = os.environ.get("GEMINI_API_KEY")
if not GOOGLE_API_KEY:
    GOOGLE_API_KEY = getpass.getpass("Enter API key for Google Gemini: ")

# Initialize LLM and embeddings (exact same as main.py)
from langchain.chat_models import init_chat_model
from langchain_huggingface import HuggingFaceEmbeddings
import torch

llm = init_chat_model("gemini-2.5-flash", model_provider="google_genai", api_key=GOOGLE_API_KEY)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-large-instruct",
    model_kwargs={'device': device}
)

print("✅ Models initialized!")

# 3. Setup PostgreSQL (using same code as main.py)
import psycopg
from langchain_postgres.vectorstores import PGVector

# Use local PostgreSQL setup (same as main.py)
db_password = os.environ.get("POSTGRES_PASSWORD") or getpass.getpass("Enter PostgreSQL password: ")
db_user = os.environ.get("POSTGRES_USER", "postgres")
db_host = os.environ.get("POSTGRES_HOST", "localhost")
db_port = os.environ.get("POSTGRES_PORT", "5432")
db_name = os.environ.get("POSTGRES_DB", "vectordb")
collection_name = "rag_embeddings"

# Create database and enable extension (same as main.py)
conn = psycopg.connect(dbname="postgres", user=db_user, password=db_password, host=db_host, port=db_port)
conn.autocommit = True
with conn.cursor() as cursor:
    cursor.execute(f"SELECT 1 FROM pg_database WHERE datname = '{db_name}'")
    if not cursor.fetchone():
        cursor.execute(f"CREATE DATABASE {db_name}")
        print(f"Database '{db_name}' created.")
conn.close()

# Enable vector extension
conn = psycopg.connect(dbname=db_name, user=db_user, password=db_password, host=db_host, port=db_port)
with conn.cursor() as cursor:
    cursor.execute("CREATE EXTENSION IF NOT EXISTS vector;")
conn.close()

connection_string = f"postgresql+psycopg://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"

# 4. Create documents using LangChain Document class
from langchain_core.documents import Document



documents = [Document(
    page_content=f"Represent this document for semantic search: {hindi_text}",
    metadata={"source": "hindi_doc", "language": "hindi"}
)]

# 5. Split documents using LangChain text splitter (same as main.py)
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""]
)

splits = text_splitter.split_documents(documents)
print(f"📄 Created {len(splits)} document chunks")

# 6. Create PostgreSQL vector store (same as main.py)
vectorstore = PGVector(
    embeddings=embeddings,
    connection=connection_string,
    collection_name=collection_name,
    use_jsonb=True
)

# Add documents to vector store
vectorstore.add_documents(splits)
print("✅ Vector store created with PostgreSQL")

# 7. Create retriever using LangChain built-in function
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print("✅ Retriever created")

# 8. Create prompt template (exact same as main.py)
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. You answer questions based only on the provided context. If the answer is not in the context, say 'I don't know'. Respond in the same language as the question."),
    ("user", "Question: {question}\n\nContext: {context}")
])

# 9. Create RAG chain using LangChain built-in functions (exact same as main.py)
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG chain created using LangChain built-ins")

# 10. Test the system
print("\n🇮🇳 HINDI RAG SYSTEM READY!")
print("="*50)

# Test questions
test_questions = [
    "भारत में कितने राज्य और केंद्र शासित प्रदेश हैं?",
    "भारत की राजधानी और सबसे बड़ा शहर कौन-सा है?",
    "भारत में कुल कितनी आधिकारिक भाषाएँ हैं?",
    "भारत से उत्पन्न चार प्रमुख धर्म कौन-से हैं?",
    "सिंधु घाटी सभ्यता क्यों महत्वपूर्ण मानी जाती है?",
    "भारत की पाँच प्रमुख नदियों के नाम लिखिए।",
    "Which sector contributes the most to India's GDP?",
    "Name two major achievements of ISRO.",
    "भारत के तीन शास्त्रीय नृत्यों के नाम लिखिए।",
    "What does 'Unity in Diversity' mean in the context of India?"
]


for question in test_questions:
    print(f"\n❓ {question}")
    try:
        response = ""
        for chunk in rag_chain.stream(question):
            response += chunk
        print(f"✅ {response}")
    except Exception as e:
        print(f"❌ Error: {e}")
    print("-" * 40)

# 11. Interactive mode (same as main.py)
# print("\n🤖 Interactive Mode - Ask your questions!")
# print("Type 'exit' to quit")

# while True:
#     question = input("\n❓ Question: ").strip()
#     if question.lower() in ['exit', 'quit', 'बाहर निकलें']:
#         print("👋 Goodbye!")
#         break
#     if question:
#         try:
#             print("\nThinking...\n")
#             response = ""
#             for chunk in rag_chain.stream(question):
#                 print(chunk, end="", flush=True)
#                 response += chunk
#             print("\n")
#         except Exception as e:
#             print(f"❌ Error: {e}")

✅ Models initialized!
📄 Created 5 document chunks
✅ Vector store created with PostgreSQL
✅ Retriever created
✅ RAG chain created using LangChain built-ins

🇮🇳 HINDI RAG SYSTEM READY!

❓ भारत में कितने राज्य और केंद्र शासित प्रदेश हैं?
✅ भारत में 28 राज्य और 8 केंद्र शासित प्रदेश हैं।
----------------------------------------

❓ भारत की राजधानी और सबसे बड़ा शहर कौन-सा है?
✅ भारत की राजधानी नई दिल्ली है जबकि मुंबई को देश का सबसे बड़ा शहर माना जाता है।
----------------------------------------

❓ भारत में कुल कितनी आधिकारिक भाषाएँ हैं?
✅ संविधान द्वारा 22 भाषाओं को आधिकारिक दर्जा दिया गया है।
----------------------------------------

❓ भारत से उत्पन्न चार प्रमुख धर्म कौन-से हैं?
✅ भारत से उत्पन्न चार प्रमुख धर्म हिंदू धर्म, बौद्ध धर्म, जैन धर्म और सिख धर्म हैं।
----------------------------------------

❓ सिंधु घाटी सभ्यता क्यों महत्वपूर्ण मानी जाती है?
✅ सिंधु घाटी सभ्यता दुनिया की सबसे प्राचीन शहरी सभ्यताओं में से एक थी।
----------------------------------------

❓ भारत की पाँच प्रमुख नदियो

In [11]:
# Add this after line 404 (after the existing test_questions loop)

# 12. Multilingual Test - Indian Languages + International
print("\n🌍 MULTILINGUAL TEST ON HINDI CONTENT")
print("="*60)
print("Testing cross-language capabilities:")
print("📄 Document: Hindi content")
print("❓ Questions: Multiple languages")
print("✅ Expected: Answers in question language")

multilingual_questions = [
    # Hindi (same language as content)
    "भारत में कितने राज्य हैं?",
    "भारत में कौन सा खेल सबसे लोकप्रिय है?",
    
    # English  
    "How many states are in India?",
    "What is the most popular sport in India?",
    
    # Bengali (বাংলা)
    "ভারতে কতটি রাজ্য আছে?",
    "ভারতে সবচেয়ে জনপ্রিয় খেলা কী?",
    
    # Tamil (தமிழ்)
    "இந்தியாவில் எத்தனை மாநிலங்கள் உள்ளன?",
    "இந்தியாவில் மிகவும் பிரபலமான விளையாட்டு எது?",
    
    # Telugu (తెలుగు)
    "భారతదేశంలో ఎన్ని రాష్ట్రాలు ఉన్నాయి?",
    "భారతదేశంలో అత్యంత ప్రాచుర్యం పొందిన క్రీడ ఏది?",
    
    # Marathi (मराठी)
    "भारतात किती राज्ये आहेत?",
    "भारतातील सर्वात लोकप्रिय खेळ कोणता आहे?",
    
    # Gujarati (ગુજરાતી)
    "ભારતમાં કેટલા રાજ્યો છે?",
    "ભારતમાં સૌથી લોકપ્રિય રમત કઈ છે?",
    
    # Punjabi (ਪੰਜਾਬੀ)
    "ਭਾਰਤ ਵਿੱਚ ਕਿੰਨੇ ਰਾਜ ਹਨ?",
    "ਭਾਰਤ ਵਿੱਚ ਸਭ ਤੋਂ ਮਸ਼ਹੂਰ ਖੇਡ ਕਿਹੜੀ ਹੈ?",
    
    # Kannada (ಕನ್ನಡ)
    "ಭಾರತದಲ್ಲಿ ಎಷ್ಟು ರಾಜ್ಯಗಳಿವೆ?",
    "ಭಾರತದಲ್ಲಿ ಅತ್ಯಂತ ಜನಪ್ರಿಯ ಆಟ ಯಾವುದು?",
    
    # Malayalam (മലയാളം)
    "ഇന്ത്യയിൽ എത്ര സംസ്ഥാനങ്ങളുണ്ട്?",
    "ഇന്ത്യയിലെ ഏറ്റവും ജനപ്രിയമായ കായികവിനോദം എന്താണ്?",
    
    # Odia (ଓଡ଼ିଆ)
    "ଭାରତରେ କେତୋଟି ରାଜ୍ୟ ଅଛି?",
    "ଭାରତରେ ସବୁଠାରୁ ଲୋକପ୍ରିୟ ଖେଳ କ'ଣ?",
    
    # International Languages
    "¿Cuántos estados hay en la India?",  # Spanish
    "Combien d'états y a-t-il en Inde?",  # French
    "Quantos estados existem na Índia?",  # Portuguese
    "インドには何の州がありますか？",  # Japanese
    "인도에는 몇 개의 주가 있습니까?",  # Korean
]

# Language labels for better output
language_labels = [
    "🇮🇳 Hindi", "🇮🇳 Hindi",
    "🇺🇸 English", "🇺🇸 English", 
    "🇧🇩 Bengali", "🇧🇩 Bengali",
    "🇮🇳 Tamil", "🇮🇳 Tamil",
    "🇮🇳 Telugu", "🇮🇳 Telugu", 
    "🇮🇳 Marathi", "🇮🇳 Marathi",
    "🇮🇳 Gujarati", "🇮🇳 Gujarati",
    "🇮🇳 Punjabi", "🇮🇳 Punjabi",
    "🇮🇳 Kannada", "🇮🇳 Kannada",
    "🇮🇳 Malayalam", "🇮🇳 Malayalam",
    "🇮🇳 Odia", "🇮🇳 Odia",
    "🇪🇸 Spanish", "🇫🇷 French", "🇵🇹 Portuguese", "🇯🇵 Japanese", "🇰🇷 Korean"
]

for i, question in enumerate(multilingual_questions):
    lang_label = language_labels[i] if i < len(language_labels) else "🌍 Unknown"
    print(f"\n{lang_label} ❓ {question}")
    
    try:
        response = ""
        for chunk in rag_chain.stream(question):
            response += chunk
        print(f"✅ {response}")
    except Exception as e:
        print(f"❌ Error: {e}")
    print("-" * 50)

print("\n🎯 MULTILINGUAL TEST COMPLETED!")
print("📊 Summary:")
print("   • Hindi content successfully processed")
print("   • Questions in 12+ languages tested")
print("   • Gemini handled cross-language translation")
print("   • Your embedding model supports multilingual queries")

# 13. Interactive mode with language detection
print("\n🤖 MULTILINGUAL Interactive Mode!")
print("Ask questions in any language - Hindi, English, Tamil, Bengali, etc.")
print("Type 'exit' or 'बाहर निकलें' or 'বের হও' to quit")

while True:
    question = input("\n🌍 Ask in any language: ").strip()
    if question.lower() in ['exit', 'quit', 'बाहर निकलें', 'বের হও', 'बाहेर या', 'বাহির হও']:
        print("👋 Goodbye! | अलविदा! | বিদায়! | अल्विदा!")
        break
    if question:
        try:
            print("\n🤔 Processing...\n")
            response = ""
            for chunk in rag_chain.stream(question):
                print(chunk, end="", flush=True)
                response += chunk
            print("\n")
        except Exception as e:
            print(f"❌ Error: {e}")


🌍 MULTILINGUAL TEST ON HINDI CONTENT
Testing cross-language capabilities:
📄 Document: Hindi content
❓ Questions: Multiple languages
✅ Expected: Answers in question language

🇮🇳 Hindi ❓ भारत में कितने राज्य हैं?
✅ भारत में 28 राज्य हैं।
--------------------------------------------------

🇮🇳 Hindi ❓ भारत में कौन सा खेल सबसे लोकप्रिय है?
✅ भारत में क्रिकेट सबसे लोकप्रिय खेल है।
--------------------------------------------------

🇺🇸 English ❓ How many states are in India?
✅ भारत में 28 राज्य हैं।
--------------------------------------------------

🇺🇸 English ❓ What is the most popular sport in India?
✅ भारत में क्रिकेट सबसे लोकप्रिय खेल है।
--------------------------------------------------

🇧🇩 Bengali ❓ ভারতে কতটি রাজ্য আছে?
✅ ভারতে ২৮টি রাজ্য আছে।
--------------------------------------------------

🇧🇩 Bengali ❓ ভারতে সবচেয়ে জনপ্রিয় খেলা কী?
✅ ভারত में क्रिकेट सबसे लोकप्रिय खेल है।
--------------------------------------------------

🇮🇳 Tamil ❓ இந்தியாவில் எத்தனை மாநிலங்கள் உள்ளன?
✅ இந்